# cognitive_alpha_position_model_v1

qlib-native CogAlpha evolution to LightGBM position-model diagnostic card.

In [1]:

from __future__ import annotations

import copy
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd
from lightgbm import LGBMClassifier
from qlib.data import D
from qlib.workflow import R

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if ROOT.as_posix() not in sys.path:
    sys.path.insert(0, ROOT.as_posix())

from src.data import QlibProviderConfig, build_alpha158_legacy_dataset, init_qlib_provider
from src.factor import (
    build_canary_sprint_frame,
    build_canary_trend_component,
    build_tsmom_stop_component,
    CognitiveAlphaCandidate,
    CognitiveAlphaFactorMiningSpec,
    materialize_cognitive_alpha_features,
    run_cognitive_alpha_factor_mining,
)
from src.workflow import (
    CognitiveAlphaEvolutionSpec,
    CognitiveAlphaGoalGateSpec,
    CognitiveAlphaGoalRunner,
    CognitiveAlphaGoalRunnerSpec,
    CognitiveAlphaPositionSpec,
    CognitiveAlphaWeightedFactorSearchSpec,
    build_cognitive_alpha_code_candidate_provider,
    codex_cli_readiness,
    run_cognitive_alpha_position_model,
    run_cognitive_alpha_weighted_factor_search,
)

CARD = ROOT / "research" / "cognitive_alpha_position_model_v1"
EXPERIMENT = "cognitive_alpha_position_model_v1"
PROVIDER_URI = ROOT / "data" / "qlib" / "soxl"
RECORDER_URI = CARD / "qlib_recorder"
SOURCE_CARD = ROOT / "research" / "agentic_alpha_cognitive_alpha_soxl_v1" / "summary.json"
FACTOR_CARD = ROOT / "research" / "agentic_alpha_ml_factor_enhanced_throttle_v1" / "summary.json"
MANIFEST = json.loads((PROVIDER_URI / "qlib_store_manifest.json").read_text(encoding="utf-8"))
SOURCE_SUMMARY = json.loads(SOURCE_CARD.read_text(encoding="utf-8"))
FACTOR_SUMMARY = json.loads(FACTOR_CARD.read_text(encoding="utf-8"))
AS_OF = MANIFEST["calendar_end"]
TRAIN = ("2019-01-02", "2024-12-31")
VALID = ("2025-01-02", "2025-12-31")
TEST = ("2026-01-02", AS_OF)

CARD.mkdir(parents=True, exist_ok=True)
RECORDER_URI.mkdir(parents=True, exist_ok=True)
config = QlibProviderConfig(provider_uri=PROVIDER_URI, region=MANIFEST["region"], market=MANIFEST["market"], kernels=1)
provider_meta = init_qlib_provider(config)
codex_runtime = codex_cli_readiness(timeout_seconds=30.0)

fields = ["$open", "$high", "$low", "$close", "$volume", "$vix"]
qlib_feature_panel = D.features(["SOXL", "SOXX"], fields, start_time=TRAIN[0], end_time=AS_OF, freq="day")
frame = qlib_feature_panel.reset_index().sort_values(["datetime", "instrument"]).reset_index(drop=True)
frame = frame.rename(columns={field: field.removeprefix("$") for field in fields})
frame = frame.loc[frame["instrument"].astype(str).str.upper() == "SOXL"].copy()
frame["datetime"] = pd.to_datetime(frame["datetime"]).dt.normalize()
close = pd.to_numeric(frame["close"], errors="coerce")
frame["label_return_5d"] = close.pct_change(5).shift(-5).ffill().fillna(0.0)
frame["forward_return"] = close.pct_change().shift(-1).fillna(0.0)

alpha_dataset = build_alpha158_legacy_dataset(
    config,
    instruments=["SOXL"],
    label="Ref($close, -5) / $close - 1",
    train=TRAIN,
    valid=VALID,
    test=TEST,
)
alpha_frames = [alpha_dataset.prepare(segment, col_set="feature") for segment in ["train", "valid", "test"]]
alpha_frame = pd.concat(alpha_frames).reset_index()
alpha_frame["datetime"] = pd.to_datetime(alpha_frame["datetime"]).dt.normalize()
preferred_alpha = ["MA5", "MA10", "MA20", "MA30", "MA60", "MAX20", "MAX30", "VIX", "SOXL_RET_5", "VOLUME_RATIO_5"]
alpha_cols = [column for column in preferred_alpha if column in alpha_frame.columns]
alpha_subset = alpha_frame[["datetime", "instrument", *alpha_cols]].rename(
    columns={column: f"alpha158_{column}" for column in alpha_cols}
)
alpha158_columns = [f"alpha158_{column}" for column in alpha_cols]
frame = frame.merge(alpha_subset, on=["datetime", "instrument"], how="left")
mining_frame = frame.loc[frame["datetime"] <= pd.Timestamp(VALID[1])].copy()

leaky_candidate = CognitiveAlphaCandidate(
    name="factor_future_leak_demo",
    level="geometric_fusion",
    agent="AgentCreative",
    theme="future leakage repair smoke test",
    required_columns=("close",),
    lookback=1,
    declared_lookahead=1,
    code="df['close'].shift(-1) / df['close'] - 1",
    economic_rationale="Invalid future leak included only to prove the quality checker and repair path.",
    function=lambda data: data["close"].shift(-1) / data["close"] - 1.0,
)
candidate_provider = build_cognitive_alpha_code_candidate_provider()

R.set_uri(RECORDER_URI.as_posix())
with tempfile.TemporaryDirectory(prefix="cogalpha_position_") as tmp_dir:
    evolution_spec = CognitiveAlphaEvolutionSpec(
        run_id="cognitive_alpha_evolution",
        profile="soxl_full",
        runtime_dir=Path(tmp_dir),
        generations=24,
        parent_pool_size=32,
        children_per_generation=96,
        initial_pool_size=80,
        qualified_percentile=0.65,
        elite_percentile=0.80,
        max_abs_correlation=0.95,
        min_observations=180,
        max_missing_ratio=0.55,
        min_distinct_values=5,
        fallback_top_n=8,
    )
    with R.start(experiment_name=EXPERIMENT):
        recorder = R.get_recorder()
        R.log_params(
            provider_uri=provider_meta["provider_uri"],
            market=MANIFEST["market"],
            region=MANIFEST["region"],
            instrument="SOXL",
            train_start=TRAIN[0],
            train_end=TRAIN[1],
            valid_start=VALID[0],
            valid_end=VALID[1],
            test_start=TEST[0],
            test_end=TEST[1],
            model_family="multi_model_lightgbm_logistic_ridge_histgbdt",
            alpha158_feature_count=alpha_dataset.qlib_feature_set["feature_count"],
            candidate_provider=candidate_provider.provider_id,
        )
        factor_mining = run_cognitive_alpha_factor_mining(
            mining_frame,
            spec=CognitiveAlphaFactorMiningSpec(
                evolution_spec=evolution_spec,
                label_col="label_return_5d",
                alpha158_columns=tuple(alpha158_columns),
            ),
            initial_candidates=[leaky_candidate],
            candidate_provider=candidate_provider,
        )
        evolution = factor_mining.evolution_result
        runtime_run_dir = factor_mining.run_dir
        for checkpoint_json in [runtime_run_dir / "state.json", runtime_run_dir / "summary.json"]:
            payload = json.loads(checkpoint_json.read_text(encoding="utf-8"))
            payload["run_dir"] = "recorder:artifacts/cognitive_alpha_evolution"
            if "spec" in payload:
                payload["spec"]["runtime_dir"] = "recorder:artifacts"
            checkpoint_json.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
        for artifact in sorted(runtime_run_dir.rglob("*")):
            if artifact.is_file():
                rel_parent = artifact.relative_to(runtime_run_dir).parent.as_posix()
                artifact_path = "cognitive_alpha_evolution" if rel_parent == "." else f"cognitive_alpha_evolution/{rel_parent}"
                recorder.log_artifact(artifact.as_posix(), artifact_path=artifact_path)

        selected_features = materialize_cognitive_alpha_features(
            frame,
            factor_mining.selected_candidates,
            selected_names=factor_mining.selected_feature_names,
        )
        soxl_return_1d = pd.to_numeric(frame["close"], errors="coerce").pct_change().fillna(0.0)
        base_model_features = frame[["datetime", *alpha158_columns, "vix"]].reset_index(drop=True).copy()
        base_model_features["realized_vol_20"] = soxl_return_1d.rolling(20, min_periods=5).std().fillna(0.02) * (252 ** 0.5)
        model_feature_cols = list(dict.fromkeys([*factor_mining.selected_feature_names, *alpha158_columns, "vix", "realized_vol_20"]))
        model_frame = pd.concat(
            [
                frame[["datetime", "forward_return", "label_return_5d"]].reset_index(drop=True),
                selected_features.reset_index(drop=True),
                base_model_features.drop(columns=["datetime"]).reset_index(drop=True),
            ],
            axis=1,
        )
        report = SOURCE_SUMMARY["cognitive_alpha_throttle_report"]
        legacy = report["legacy_soxlbacktest_benchmark"]
        constrained = report["constrained_weight_reference"]
        benchmarks = [
            {
                "name": "soxl_backtest",
                "return_pct": legacy["rulebase_return_pct"],
                "max_drawdown_pct": -abs(float(legacy["rulebase_max_drawdown_pct"])),
                "trade_count": legacy["trade_count"],
                "source": "soxl_backtest.reference",
            },
            {
                "name": "constrained_weight_reference",
                "return_pct": constrained["return_pct"],
                "max_drawdown_pct": constrained["max_drawdown_pct"],
                "trade_count": constrained["trade_count"],
                "source": "research/agentic_alpha_cognitive_alpha_soxl_v1",
            },
            {
                "name": "soxl_buyhold",
                "return_pct": legacy["soxl_buyhold_return_pct"],
                "max_drawdown_pct": 0.0,
                "source": "soxl_close_buyhold",
            },
            {
                "name": "previous_factor_enhanced",
                "return_pct": FACTOR_SUMMARY["selected_candidate"]["test"]["return_pct"],
                "max_drawdown_pct": FACTOR_SUMMARY["selected_candidate"]["test"]["max_drawdown_pct"],
                "trade_count": FACTOR_SUMMARY["selected_candidate"]["test"]["trade_count"],
                "source": "agentic_alpha_ml_factor_enhanced_throttle_v1",
            },
        ]
        canary_features = build_canary_sprint_frame(qlib_feature_panel, start=TRAIN[0], end=AS_OF)
        direct_component_frame = pd.concat(
            [
                build_canary_trend_component(
                    canary_features,
                    component="mined_canary_trend60_vix26_autocorr_h3_s1.0",
                    start=TRAIN[0],
                    end=AS_OF,
                    trend_threshold=0.06,
                    vix_max=26.0,
                    hold_days=3,
                    require_positive_autocorr=True,
                    scale=1.0,
                ),
                build_tsmom_stop_component(
                    qlib_feature_panel,
                    component="mined_tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02",
                    start=TRAIN[0],
                    end=AS_OF,
                    score_name="soxx_ret10",
                    threshold=0.0,
                    regime_filter="lowhv",
                    scale=0.75,
                    stop_loss=0.02,
                    lowhv_quantile=0.70,
                ),
                build_tsmom_stop_component(
                    qlib_feature_panel,
                    component="mined_tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03",
                    start=TRAIN[0],
                    end=AS_OF,
                    score_name="soxx_ret252",
                    threshold=0.15,
                    regime_filter="vix30",
                    scale=0.75,
                    stop_loss=0.03,
                    lowhv_quantile=0.70,
                ),
            ],
            ignore_index=True,
        )
        direct_exposure = direct_component_frame.pivot_table(
            index="datetime", columns="component", values="exposure", aggfunc="sum"
        ).sort_index().fillna(0.0)
        direct_returns = frame[["datetime", "close"]].copy().sort_values("datetime")
        direct_returns["forward_return"] = pd.to_numeric(direct_returns["close"], errors="coerce").pct_change().shift(-1)
        direct_frame = direct_exposure.reset_index().merge(
            direct_returns[["datetime", "forward_return"]].dropna(subset=["forward_return"]),
            on="datetime",
            how="inner",
        )
        direct_frame = direct_frame.loc[
            (direct_frame["datetime"] >= pd.Timestamp(TEST[0])) & (direct_frame["datetime"] <= pd.Timestamp(TEST[1]))
        ].copy()
        constrained_direct_weights = {
            "mined_canary_trend60_vix26_autocorr_h3_s1.0": constrained["weights"].get("canary_trend60_vix26_autocorr_h3_s1.0", 0.0),
            "mined_tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02": constrained["weights"].get("tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02", 0.0),
            "mined_tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03": constrained["weights"].get("tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03", 0.0),
        }
        direct_result = run_cognitive_alpha_weighted_factor_search(
            direct_frame,
            spec=CognitiveAlphaWeightedFactorSearchSpec(
                component_cols=list(direct_exposure.columns),
                candidates=[
                    {
                        "candidate": "mined_canary_only_budget_h01_direct",
                        "weights": {"mined_canary_trend60_vix26_autocorr_h3_s1.0": 1.7732},
                        "policy": {"min_position": 0.0, "max_position": 1.0, "rebalance_threshold": 0.10},
                    },
                    {
                        "candidate": "mined_canary_tsmom_signed_budget_h03_direct",
                        "weights": {
                            "mined_canary_trend60_vix26_autocorr_h3_s1.0": 1.8,
                            "mined_tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02": -0.1,
                            "mined_tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03": 0.1,
                        },
                        "policy": {"min_position": 0.0, "max_position": 1.0, "rebalance_threshold": 0.30},
                    },
                    {
                        "candidate": "stored_constrained_reference_recomputed",
                        "weights": constrained_direct_weights,
                        "policy": constrained["policy"],
                    },
                ],
                min_effective_component_count=2,
                constrained_reference_name="constrained_weight_reference",
                max_drawdown_floor=-20.0,
            ),
            benchmarks=benchmarks,
        )
        position_spec = CognitiveAlphaPositionSpec(
            feature_cols=model_feature_cols,
            train=TRAIN,
            validation=VALID,
            test=TEST,
            model_specs=[
                {"model": "lgbm_classifier", "positive_threshold": 0.0},
                {"model": "logistic_regression", "positive_threshold": 0.0},
                {"model": "ridge", "task": "regression"},
                {"model": "hist_gradient_boosting_classifier", "positive_threshold": 0.0},
            ],
            position_specs=[
                {
                    "position_mode": "probability",
                    "threshold": 0.45,
                    "scale": 1.0,
                    "max_position": 1.0,
                    "rebalance_threshold": 0.20,
                    "max_daily_change": 0.50,
                    "min_hold_days": 3,
                    "target_vol": 0.35,
                    "realized_vol_col": "realized_vol_20",
                    "min_vol_scale": 0.0,
                    "max_vol_scale": 1.0,
                    "risk_gate_col": "vix",
                    "risk_gate_threshold": 30.0,
                    "risk_gate_direction": "above",
                    "risk_gate_max_position": 0.50,
                    "objective_max_drawdown_floor": -25.0,
                    "objective_return_weight": 0.25,
                    "objective_return_to_drawdown_weight": 1.0,
                    "turnover_penalty": 0.5,
                    "reversal_penalty": 4.0,
                    "trade_penalty": 0.05,
                },
                {
                    "position_mode": "binary",
                    "threshold": 0.55,
                    "scale": 1.0,
                    "max_position": 1.0,
                    "rebalance_threshold": 0.25,
                    "max_daily_change": 0.50,
                    "min_hold_days": 3,
                    "target_vol": 0.35,
                    "realized_vol_col": "realized_vol_20",
                    "min_vol_scale": 0.0,
                    "max_vol_scale": 1.0,
                    "risk_gate_col": "vix",
                    "risk_gate_threshold": 30.0,
                    "risk_gate_direction": "above",
                    "risk_gate_max_position": 0.50,
                    "objective_max_drawdown_floor": -25.0,
                    "objective_return_weight": 0.25,
                    "objective_return_to_drawdown_weight": 1.0,
                    "turnover_penalty": 0.5,
                    "reversal_penalty": 4.0,
                    "trade_penalty": 0.05,
                },
            ],
            min_validation_trade_count=1,
            max_validation_reversal_ratio=0.6,
            validation_max_drawdown_floor=-20.0,
            validation_max_daily_loss_floor=-12.0,
            max_validation_total_turnover=30.0,
            test_max_drawdown_floor=-20.0,
            test_max_daily_loss_floor=-12.0,
            max_test_total_turnover=20.0,
        )
        position_result = run_cognitive_alpha_position_model(model_frame, spec=position_spec, benchmarks=benchmarks)
        upload_dir = Path(tmp_dir) / "position_uploads"
        upload_dir.mkdir(parents=True, exist_ok=True)
        positions_path = upload_dir / "positions.csv"
        position_result.positions.to_csv(positions_path, index=False)
        recorder.log_artifact(positions_path.as_posix(), artifact_path="cognitive_alpha_position")
        direct_positions_path = upload_dir / "direct_weighted_positions.csv"
        direct_result.positions.to_csv(direct_positions_path, index=False)
        recorder.log_artifact(direct_positions_path.as_posix(), artifact_path="cognitive_alpha_direct_weighted")
        R.log_metrics(
            selected_test_return_pct=float(position_result.summary["selected_candidate"]["test"]["return_pct"]),
            selected_test_max_drawdown_pct=float(position_result.summary["selected_candidate"]["test"]["max_drawdown_pct"]),
            direct_factor_return_pct=float(direct_result.summary["selected_candidate"]["return_pct"]),
            direct_factor_beats_constrained=float(direct_result.summary["comparison"]["beats_constrained_weight_reference"]),
            latest_target_position=float(position_result.latest_signal["target_position"]),
        )
        recorder_id = recorder.id
        recorder_local_dir = Path(recorder.get_local_dir())

for transient in ["code_cached.txt", "code_diff.txt", "code_status.txt"]:
    transient_path = recorder_local_dir / "artifacts" / transient
    if transient_path.exists():
        transient_path.unlink()

evolution_summary = copy.deepcopy(evolution.summary)
evolution_summary["run_dir"] = "recorder:artifacts/cognitive_alpha_evolution"
evolution_summary["spec"]["runtime_dir"] = "recorder:artifacts"
factor_mining_summary = copy.deepcopy(factor_mining.summary)
factor_mining_summary["evolution"] = evolution_summary

sidecars = {
    "model_selection": "model_selection.csv",
    "benchmark_comparison": "backtest_benchmark_comparison.csv",
    "position_summary": "position_summary.json",
    "latest_signal": "latest_signal.json",
    "direct_factor_candidate_metrics": "direct_factor_candidate_metrics.csv",
    "direct_factor_benchmark_comparison": "direct_factor_benchmark_comparison.csv",
    "direct_factor_summary": "direct_factor_summary.json",
    "failure_modes": "failure_modes.md",
    "goal_manifest": "goal_manifest.json",
}
position_result.metrics.to_csv(CARD / sidecars["model_selection"], index=False)
position_result.benchmark_comparison.to_csv(CARD / sidecars["benchmark_comparison"], index=False)
(CARD / sidecars["position_summary"]).write_text(json.dumps(position_result.summary, indent=2, sort_keys=True), encoding="utf-8")
(CARD / sidecars["latest_signal"]).write_text(json.dumps(position_result.latest_signal, indent=2, sort_keys=True), encoding="utf-8")
direct_result.metrics.to_csv(CARD / sidecars["direct_factor_candidate_metrics"], index=False)
direct_result.benchmark_comparison.to_csv(CARD / sidecars["direct_factor_benchmark_comparison"], index=False)
(CARD / sidecars["direct_factor_summary"]).write_text(json.dumps(direct_result.summary, indent=2, sort_keys=True), encoding="utf-8")
direct_benchmark_pass = bool(direct_result.summary["comparison"]["beats_constrained_weight_reference"])
promotion_decision = position_result.summary["promotion_decision"]


def _goal_data_gate():
    return [
        {
            "source": "qlib.data.D.features",
            "provider_uri": PROVIDER_URI.resolve().as_posix(),
            "market": MANIFEST["market"],
            "region": MANIFEST["region"],
            "max_datetime": AS_OF,
            "calendar_start": provider_meta["calendar_start"],
            "calendar_end": provider_meta["calendar_end"],
            "fields": fields,
            "instruments": ["SOXL"],
        }
    ]


def _goal_evolution(_manifests):
    return evolution_summary


def _goal_position(_manifests, _evolution_result):
    return position_result.summary


def _goal_direct_factor_backtest(_manifests, _evolution_result, _position_result):
    return direct_result.summary


def _goal_research_card(_manifests, _evolution_result, _position_result):
    return {
        "card_dir": CARD.relative_to(ROOT).as_posix(),
        "recorder_id": recorder_id,
        "recorder_local_dir": recorder_local_dir.as_posix(),
        "promotion_decision": promotion_decision,
    }


def _goal_signal_push(_manifests, _model_results):
    return [
        {
            "source": "qlib_recorder.latest_signal",
            "action": position_result.latest_signal["action"],
            "as_of": position_result.latest_signal["as_of"],
            "target_position": position_result.latest_signal["target_position"],
            "target_percent": position_result.latest_signal["target_percent"],
            "real_broker_order": False,
        }
    ]


goal_runner = CognitiveAlphaGoalRunner(
    CognitiveAlphaGoalRunnerSpec(
        run_id="cognitive_alpha_goal_runner",
        objective="SOXL CogAlpha data/evolution/position/card/signal goal chain",
        state_path=CARD / "goal_state.json",
        manifest_path=CARD / sidecars["goal_manifest"],
        required_max_datetime=AS_OF,
        resume=False,
        gates=CognitiveAlphaGoalGateSpec(
            min_initial_candidate_count=80,
            min_completed_generations=24,
            min_selected_count=1,
            min_repair_attempt_count=1,
            required_candidate_provider_ids=("deterministic_code_agent_pool",),
            required_candidate_provider_kinds=("repo_owned_code",),
            require_candidate_provider_no_openai_api_key=True,
            required_metric_names=("ic", "icir", "rank_ic", "rank_icir", "mutual_information"),
            required_evolution_ops=("mutation", "crossover", "crossover_mutation"),
            required_position_model_families=("lgbm_classifier", "logistic_regression", "ridge", "hist_gradient_boosting_classifier"),
            required_benchmark_names=(
                "soxl_backtest",
                "constrained_weight_reference",
                "soxl_buyhold",
                "previous_factor_enhanced",
            ),
            required_direct_factor_benchmark_names=(
                "constrained_weight_reference",
                "soxl_backtest",
                "soxl_buyhold",
                "previous_factor_enhanced",
            ),
            require_direct_factor_beats_constrained_weight_reference=False,
        ),
    ),
    data_gate=_goal_data_gate,
    evolution=_goal_evolution,
    position=_goal_position,
    direct_factor_backtest=_goal_direct_factor_backtest,
    research_card=_goal_research_card,
    signal_push=_goal_signal_push,
)
goal_manifest = goal_runner.run()
(CARD / "goal_state.json").unlink(missing_ok=True)
(CARD / sidecars["failure_modes"]).write_text(
    "# Failure Modes\n\n"
    "- 这是 CogAlpha evolution -> multi-model position 的诊断卡，不是晋升卡。\n"
    "- goal_manifest.json 记录 data_gate、candidate_generation、quality_checker、fitness_evaluation、evolution_loop、model_and_position、research_card、signal_push 的 goal-mode stage 证据。\n"
    "- 输入 alpha 来自 soxl_full evolution pool，已跑 21-agent/80/96/24 generation full profile。\n"
    "- direct_factor_backtest 是因子直连受约束仓位搜索，只能作为 benchmark/prior，不能作为最终晋升门。\n"
    "- 最终晋升必须由 position_model 或 production paper/sim 通过；当前卡只保留为 research evidence。\n"
    "- 本研究面向交易复盘和风险控制，不构成买卖建议。\n",
    encoding="utf-8",
)
(CARD / "workflow.yaml").write_text(
    "\n".join(
        [
            f"experiment: {EXPERIMENT}",
            "card_type: diagnostic_cognitive_alpha_position_model",
            "paper_reference:",
            "  arxiv_id: '2511.18850'",
            "  method: checkpointed_evolution_multimodel_position",
            "qlib:",
            f"  provider_uri: {PROVIDER_URI.resolve().as_posix()}",
            f"  recorder_uri: {RECORDER_URI.resolve().as_posix()}",
            f"  market: {MANIFEST['market']}",
            f"  region: {MANIFEST['region']}",
            "segments:",
            f"  train: [{TRAIN[0]}, {TRAIN[1]}]",
            f"  validation: [{VALID[0]}, {VALID[1]}]",
            f"  test: [{TEST[0]}, {TEST[1]}]",
            "factor_mining:",
            "  workflow: cognitive_alpha_factor_mining",
            "  source_layer: src.factor",
            "  candidate_scope: all_generations",
            "  output: alpha_factor_library",
            "  position_model_included: false",
            "  position_signal_included: false",
            "model_selection:",
            "  models: [lgbm_classifier, logistic_regression, ridge, hist_gradient_boosting_classifier]",
            "  selected_by: validation_selection_objective_score",
            "  validation_max_drawdown_floor: -20.0",
            "  test_max_drawdown_floor: -20.0",
            "direct_factor_backtest:",
            "  workflow: cognitive_alpha_weighted_factor_search",
            "  selected_by: diagnostic_return_sorted_long_only_gate",
            "  recorder_positions: artifacts/cognitive_alpha_direct_weighted/direct_weighted_positions.csv",
            "artifact_policy:",
            "  policy: slim_card_recorder_first",
            "  workflow_config: workflow.yaml",
            "  machine_entrypoint: summary.json",
            "",
        ]
    ),
    encoding="utf-8",
)
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob("*") if path.is_file())
summary = {
    "experiment": EXPERIMENT,
    "card_type": "diagnostic_cognitive_alpha_position_model",
    "workflow_config": "workflow.yaml",
    "promotion_decision": promotion_decision,
    "paper_reference": {"arxiv_id": "2511.18850", "version_checked": "v3", "method": "checkpointed_evolution_multimodel_position"},
    "codex_runtime": codex_runtime,
    "provider_uri": PROVIDER_URI.resolve().as_posix(),
    "recorder_uri": RECORDER_URI.resolve().as_posix(),
    "recorder_id": recorder_id,
    "recorder_local_dir": recorder_local_dir.as_posix(),
    "recorder_artifacts": recorder_artifacts,
    "market": MANIFEST["market"],
    "region": MANIFEST["region"],
    "instrument": "SOXL",
    "as_of": AS_OF,
    "data_range": [TRAIN[0], AS_OF],
    "segments": {"train": list(TRAIN), "validation": list(VALID), "test": list(TEST)},
    "qlib_native_api": ["qlib.init", "qlib.data.D.features", "DatasetH", "DataHandlerLP", "Alpha158.get_feature_config", "qlib.workflow.R.start"],
    "qlib_docs": {"data": "https://qlib.readthedocs.io/en/latest/component/data.html", "workflow": "https://qlib.readthedocs.io/en/latest/component/workflow.html"},
    "alpha158_feature_set": alpha_dataset.qlib_feature_set,
    "evolution": evolution_summary,
    "factor_mining": factor_mining_summary,
    "direct_factor_backtest": direct_result.summary,
    "position_model": position_result.summary,
    "latest_signal": position_result.latest_signal,
    "sidecars": sidecars,
    "artifact_policy": {
        "policy": "slim_card_recorder_first",
        "primary_artifact": f"{EXPERIMENT}.ipynb",
        "machine_entrypoint": "summary.json",
        "workflow_config": "workflow.yaml",
        "kept_sidecars": sorted(sidecars.values()),
        "recorder_only_artifacts": {
            "candidate_pool": "artifacts/cognitive_alpha_evolution/candidate_pool/generation_*.jsonl",
            "quality": "artifacts/cognitive_alpha_evolution/quality/generation_*.csv",
            "fitness": "artifacts/cognitive_alpha_evolution/fitness/generation_*.csv",
            "rejected": "artifacts/cognitive_alpha_evolution/rejected/generation_*.jsonl",
            "lineage": "artifacts/cognitive_alpha_evolution/lineage/generation_*.jsonl",
            "state": "artifacts/cognitive_alpha_evolution/state.json",
            "positions": "artifacts/cognitive_alpha_position/positions.csv",
            "direct_weighted_positions": "artifacts/cognitive_alpha_direct_weighted/direct_weighted_positions.csv",
        },
    },
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
(CARD / "conclusion.md").write_text(
    "# 结论\n\n"
    "- what_ran: 执行 cognitive_alpha_position_model_v1，先通过 src.factor 的 CogAlpha factor mining 挖掘 alpha library，再把 mined factors、Alpha158、VIX 与 realized volatility 接入 LightGBM/Logistic/Ridge/HistGBDT 仓位模型，并用 qlib recorder 记录 checkpoint 和 positions。\n"
    f"- data: provider={PROVIDER_URI.resolve().as_posix()}，as_of={AS_OF}，train={TRAIN[0]} 到 {TRAIN[1]}，validation={VALID[0]} 到 {VALID[1]}，test={TEST[0]} 到 {TEST[1]}。\n"
    f"- key_metrics: selected={position_result.summary['selected_candidate']['candidate']}，test_return={position_result.summary['selected_candidate']['test']['return_pct']:.4f}%，MaxDD={position_result.summary['selected_candidate']['test']['max_drawdown_pct']:.4f}%，latest_target={position_result.latest_signal['target_percent']:.2f}%。\n"
    f"- direct_factor_backtest: selected={direct_result.summary['selected_candidate']['candidate']}，return={direct_result.summary['selected_candidate']['return_pct']:.4f}%，MaxDD={direct_result.summary['selected_candidate']['max_drawdown_pct']:.4f}%，beats_constrained={direct_benchmark_pass}，promotion={direct_result.summary['promotion_decision']}。\n"
    f"- benchmark: constrained_weight_reference={constrained['return_pct']:.4f}%，soxl_backtest={legacy['rulebase_return_pct']:.4f}%，buyhold={legacy['soxl_buyhold_return_pct']:.4f}%。\n"
    f"- promotion_decision: {promotion_decision}。direct_factor_backtest 只作为 benchmark/prior；最终晋升仅看 position_model 或 production paper/sim，本卡未进入 production。\n"
    "- risks: CogAlpha 论文核心是 factor mining；本卡的仓位层只是下游诊断，SOXL 单标的 IC/仓位模型仍可能过拟合；本研究面向交易复盘和风险控制，不构成买卖建议。\n"
    "- goal_mode: goal_manifest.json 已记录 data/evolution/position/direct_factor_backtest/card/signal 的完整 stage manifest，可作为下一轮 resume/audit 入口。\n"
    f"- next_action: 进入 validated research 复核：做 walk-forward、月度窗口、最新数据刷新和 paper-sim；qlib recorder id={recorder_id}。\n",
    encoding="utf-8",
)

print(json.dumps({"recorder_id": recorder_id, "latest_signal": position_result.latest_signal, "promotion_decision": promotion_decision}, indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[22669:MainThread](2026-06-26 17:09:12,004) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[22669:MainThread](2026-06-26 17:09:12,006) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[22669:MainThread](2026-06-26 17:09:12,006) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[22669:MainThread](2026-06-26 17:09:23,192) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[22669:MainThread](2026-06-26 17:09:23,283) INFO - qlib.workflow - [exp.py:258] - Experiment 877908106740255892 starts running ...


[22669:MainThread](2026-06-26 17:09:23,400) INFO - qlib.workflow - [recorder.py:345] - Recorder 1acde5b6d13c407da17049ac0d1afe46 starts running under Experiment 877908106740255892 ...


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[22669:MainThread](2026-06-26 17:09:38,279) INFO - qlib.timer - [log.py:127] - Time cost: 0.004s | waiting `async_log` Done


{
  "recorder_id": "1acde5b6d13c407da17049ac0d1afe46",
  "latest_signal": {
    "as_of": "2026-06-24",
    "candidate": "lgbm_classifier_probability_p0.45_s1_h0.2_hold3",
    "action": "HOLD",
    "target_position": 0.0,
    "target_percent": 0.0,
    "previous_target_position": 0.0,
    "target_delta": 0.0,
    "target_delta_percent": 0.0,
    "probability": 0.32033866223872726,
    "last_change_date": "2026-06-24",
    "last_change_target_position": 0.0
  },
  "promotion_decision": "not_promoted_under_constrained_weight_reference"
}


## 中文结论

详见 `conclusion.md`。本研究面向交易复盘和风险控制，不构成买卖建议。